In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import pprint
import pickle


# Loading The Datset 

In [5]:
wine=pd.read_csv("wine_dataset.csv")
#print(wine)
corr=wine.corr()
#plt.figure(figsize=(20,20))
#sns.heatmap(
#    corr[['target']].sort_values(by='target', ascending=False),
#    cmap='coolwarm',
#    annot=True
#)
#plt.show()

# we get to see the column alkalinity of ash, nonflavoid_phenol,malic acid
X=pd.DataFrame()
y=wine['target']
l=['alcalinity_of_ash','nonflavanoid_phenols','malic_acid']
for i in l:
    X[i]=wine[i]

# Test Train Split 

In [6]:
def test_train_split(X, y, random_state=42, test_size=0.2):
    rng = np.random.default_rng(random_state)
    
    n_samples = X.shape[0]
    indices = np.arange(n_samples)
    
    # shuffle all indices
    rng.shuffle(indices)
    
    split = int(n_samples * (1 - test_size))
    
    train_indices = indices[:split]
    test_indices = indices[split:]
    
    X_train, X_test = X.iloc[train_indices], X.iloc[test_indices]
    y_train, y_test = y.iloc[train_indices], y.iloc[test_indices]
    
    return X_train, X_test, y_train, y_test

# Standardize Data Blueprint

In [7]:

def standardize_data(X_train, X_test):
    for i in X_train.columns:
        mean = X_train[i].mean()
        std =  X_train[i].std()
        X_train[i] =  (X_train[i]-(mean))/std
        X_test[i] = (X_test[i]-(mean))/std
        print(mean,std)
    return X_train, X_test


# Linar Regression Model 


In [8]:

class linear_regresssion:
    def __init__(self):
        self.w=None
        self.b=0
    def initalize(self,n):
        self.w=np.zeros((n,1))
    def forward(self,X):
        X=np.asarray(X)
        return np.dot(X,self.w)+self.b
    def backward(self, X, fwd, y):
        

        X = np.asarray(X)
        y = np.asarray(y).reshape(-1,1)
        m = X.shape[0]
        error = fwd - y
        dw = (1/m) * np.dot(X.T, error)
        db = (1/m) * np.sum(error)
        return dw, db
    def fit(self,X,y,lr=0.001,interation=100000):
        
        self.initalize(X.shape[1])
       
        
        for i in range(interation):
            fwd=self.forward(X)
            dw ,db =self.backward(X,fwd,y)

            self.w -= lr*dw
            self.b -= lr*db
    def predict(self,X):
        return self.forward(X)
    
    def save_model(self, filename="linear_regression_wine.pkl"):
        with open(filename, "wb") as f:
            pickle.dump((self.w, self.b), f)
    
    @classmethod
    def load_model(cls, filename):
        with open(filename, "rb") as f:
            w, b = pickle.load(f)
        model = cls()
        model.w = w
        model.b = b
        return model


# Metrics For Model Evalution

In [9]:
class RegressionMetrics:
        @staticmethod
        def mean_squared_error(y_true, y_pred):
            """
            Calculate the Mean Squared Error (MSE).

            Args:
                y_true (numpy.ndarray): The true target values.
                y_pred (numpy.ndarray): The predicted target values.

            Returns:
                float: The Mean Squared Error.

            """
            print(1/len(y_true))

            return (1/len(y_true))*(np.sum(y_true-y_pred)**2)

            
        @staticmethod
        def root_mean_squared_error(y_true, y_pred):
            """
            Calculate the Root Mean Squared Error (RMSE).

            Args:
                y_true (numpy.ndarray): The true target values.
                y_pred (numpy.ndarray): The predicted target values.

            Returns:
                float: The Root Mean Squared Error.

            """
            return (RegressionMetrics.mean_squared_error(y_true,y_pred))**0.5
            
        @staticmethod
        def r_squared(y_true, y_pred):
            return 1-(np.sum((y_true-y_pred)**2)/np.sum((y_true-np.mean(y_true))**2))

# spliting and standardizeing data

In [10]:
X_train, X_test, y_train, y_test =test_train_split(X,y)

X_train, X_test = standardize_data(X_train, X_test)

19.664788732394364 3.450916153063557
0.3659859154929578 0.12596329454753635
2.325352112676056 1.1257531203169309


# training the model

In [11]:
model=linear_regresssion()
model.fit(X_train,y_train)
model.save_model()

# evalution of model



In [ ]:
#model=model.load_model("linear_regression_wine.pkl")
y_pred = model.predict(X_test)
y_pred=np.asarray(y_pred)

y_test = y_test.values.reshape(-1, 1) #as the y_test has shape (36,) and need to convert to (36,1)
sums=0
mse_value = RegressionMetrics.mean_squared_error(y_test, y_pred)
rmse_value = RegressionMetrics.root_mean_squared_error(y_test, y_pred)
r_squared_value = RegressionMetrics.r_squared(y_test, y_pred)
print(model.w,model.b)
print(y_test)
print(y_pred)

print(f"Mean Squared Error (MSE): {mse_value}")
print(f"Root Mean Squared Error (RMSE): {rmse_value}")
print(f"R-squared (Coefficient of Determination): {r_squared_value}")

0.027777777777777776
0.027777777777777776
[[0.25689259]
 [0.19750308]
 [0.17844378]] 0.9788732394365641
[[0]
 [2]
 [0]
 [2]
 [0]
 [1]
 [1]
 [2]
 [0]
 [0]
 [0]
 [2]
 [0]
 [0]
 [0]
 [0]
 [1]
 [0]
 [0]
 [1]
 [2]
 [2]
 [0]
 [1]
 [0]
 [0]
 [2]
 [1]
 [0]
 [1]
 [2]
 [0]
 [2]
 [0]
 [1]
 [2]]
[[0.62150335]
 [1.01945088]
 [0.55889921]
 [1.00980248]
 [0.61023898]
 [0.92578695]
 [0.73008189]
 [2.29479527]
 [0.60246659]
 [0.88315058]
 [0.55765136]
 [1.75392991]
 [0.7269698 ]
 [0.9399631 ]
 [0.34031047]
 [0.79317943]
 [1.04481629]
 [0.64380534]
 [0.42620081]
 [1.05728245]
 [1.65935729]
 [1.29061849]
 [1.00408418]
 [0.99330417]
 [0.52646116]
 [0.46200153]
 [1.12376489]
 [0.92236174]
 [0.35918765]
 [0.91166468]
 [1.41225515]
 [0.21697258]
 [1.04731501]
 [0.5194587 ]
 [0.44182512]
 [1.71501035]]
Mean Squared Error (MSE): 0.47746437416074133
Root Mean Squared Error (RMSE): 0.6909879696208475
R-squared (Coefficient of Determination): 0.5254953201234883
